In [337]:
import pandas as pd
import glob
from scipy import signal
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
from sklearn import preprocessing
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn import datasets, linear_model
from sklearn.model_selection import cross_val_score


In [338]:
###
# All variables releated data are located here to make an order
###
PATH="C:\\Users\\emirc\\Desktop\\AAR\\data\\csv\\datas/*" #Refer the path having the csv files
ANIMALS_USED=[1]   #Animal numbers to study on 
COLUMNS_REMOVE=['Mx','My','Mz','Gx','Gy','Gz'] #Columns to delete such as Gx Gy Gz
ACTIVITIES_MERGE={'running-natural': 'running',
                 'running-rider': 'running',
                  'trotting-natural': 'trotting',    #Having nearly the same meaning
                  'trotting-rider': 'trotting',} 
TIME_PERIODS = 200
STEP_DISTANCE = 100
filePaths = sorted(glob.glob(PATH))


In [339]:
def createDataFrame(filePaths):
    dataFrame=pd.DataFrame()
    for fileName in filePaths:
        csv=pd.read_csv(fileName,low_memory=False)
        dataFrame=dataFrame.append(csv)
    
    #Removing columns
    dataFrame.drop(COLUMNS_REMOVE,axis=1,inplace=True)
    le = preprocessing.LabelEncoder()
    dataFrame['ActivityEncoded'] = le.fit_transform(dataFrame['label'].values.ravel())

    
    #Removing needless horses
    dataFrame=dataFrame[dataFrame['subject'].isin(ANIMALS_USED)]
    
    #Merging activities
    for key in ACTIVITIES_MERGE:
        dataFrame['label'] = dataFrame['label'].replace(to_replace=key, value=ACTIVITIES_MERGE.get(key))
       
    #deleting unlabeled data
    dataFrame=dataFrame[~(dataFrame['label'].isin(['unknown']))]

    #removing nulls
    dataFrame=dataFrame.dropna(axis=0, how='any')


    return dataFrame

In [340]:
def lowBufferFiltering(dataFrame):
    sos = signal.butter(N=3, Wn=30, btype='lowpass', fs=100, output='sos')
    dataFrame['Ax'] = signal.sosfilt(sos, dataFrame['Ax'])
    dataFrame['Ay'] = signal.sosfilt(sos, dataFrame['Ay'])
    dataFrame['Az'] = signal.sosfilt(sos, dataFrame['Az'])
    return dataFrame
    

In [341]:
def windowing(dataFrame,labelName):
    windows = []
    labels = []
    for i in range(0, len(dataFrame)-TIME_PERIODS, STEP_DISTANCE):
        axs = dataFrame['Ax'].values[i: i + TIME_PERIODS]
        ays = dataFrame['Ay'].values[i: i + TIME_PERIODS]
        azs = dataFrame['Az'].values[i: i + TIME_PERIODS]
    
        # Retrieve the most often used label in this segment
        label = stats.mode(dataFrame[labelName][i: i + TIME_PERIODS])[0][0]
        windows.append([axs, ays, azs])
        labels.append(label)
        
    reshaped_windows = np.asarray(windows, dtype= np.float32).reshape(-1, TIME_PERIODS, 3)
    labels = np.asarray(labels)
    return reshaped_windows, labels

In [342]:
def kFCrossValidation(dataSet):
    x = dataFrame.iloc[:,2:1941].values
    y = dataFrame.iloc[:,0].values
    cv = KFold(n_splits=10, random_state=1, shuffle=True)
    model = RandomForestClassifier(random_state=1)
    scores = cross_val_score(model, x, y, scoring='accuracy', cv=cv, n_jobs=-1)
    print('Accuracy: %.3f (%.3f)' % (np.mean(scores), np.std(scores)))
    

In [362]:
dataFrame=createDataFrame(filePaths)

In [363]:
LABEL = 'ActivityEncoded'
le = preprocessing.LabelEncoder()
dataFrame[LABEL] = le.fit_transform(dataFrame['label'].values.ravel())

In [364]:
#filtering
dataFrame=lowBufferFiltering(dataFrame)

In [365]:
#Windowing
window, labels_=windowing(dataFrame,'Ax')

num_time_periods, num_sensors = window.shape[1], window.shape[2]
input_shape = (num_time_periods*num_sensors)
window = window.reshape(window.shape[0], input_shape)

dataFrame=pd.DataFrame(window)

In [366]:
kFCrossValidation(dataFrame)
print(dataFrame)

Accuracy: nan (nan)
           0          1          2          3          4          5    \
0     1.724824   6.399451  10.805390  11.905398  10.249974   9.739391   
1     5.923833   7.311477   6.712793   4.561272   7.750417  10.161329   
2     9.369692   8.823880  10.111292  11.597388   8.684523   4.578300   
3    10.839377   6.502428   8.094160  13.024595   8.641641   3.187019   
4     6.513785   1.263081   4.066184  11.263055  12.912168  11.148661   
..         ...        ...        ...        ...        ...        ...   
266   6.235320   6.851469   7.230922   6.312361   5.872792   5.021214   
267   8.648054   9.078161   9.342417   9.082249   8.642789   9.003624   
268  10.580711  10.416831   7.777531   7.704178   9.112560   9.511297   
269  10.373693   8.949777   8.364917   8.324034   7.964622   9.757623   
270   9.019103   7.902210   8.317863   8.717982   8.258934   7.958982   

           6          7          8          9    ...       590       591  \
0    11.408586  10.879099  

c:\users\emirc\appdata\local\programs\python\python39\lib\site-packages\sklearn\model_selection\_validation.py:372: FitFailedWarning: 
10 fits failed out of a total of 10.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
10 fits failed with the following error:
Traceback (most recent call last):
  File "c:\users\emirc\appdata\local\programs\python\python39\lib\site-packages\sklearn\model_selection\_validation.py", line 681, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\users\emirc\appdata\local\programs\python\python39\lib\site-packages\sklearn\ensemble\_forest.py", line 366, in fit
    y, expanded_class_weight = self._validate_y_class_weight(y)
  File "c:\users\emirc\appdata\local\programs\python\python39